# Data Checks — What Each Artifact Is, What It Looks Like, and Whether It Is Correct

Walks through everything in `Data/` produced by `Data_preparation.ipynb` and
`GFP_experiments.ipynb`. For each artifact: a short description, a look at the actual
content, and explicit checks. Every check is collected into a summary table at the end.

| Artifact | Role |
|---|---|
| `HI-Small_*.csv/txt` | raw inputs from Kaggle (never modified) |
| `edge_features.csv` | one row per transaction: metadata + 20 baseline + 61 GFP features (raw, pre-normalization) |
| `node_features.csv` | one row per account: entity-type one-hot (6) |
| `feature_meta.json` | column lists, dimensions, ablation grid — the contract for model notebooks |
| `standard_scaler.pkl` | train-fit normalization parameters for the GFP vertex statistics |
| `account_to_idx.pkl` | account string → integer node id |
| `train/val/test_graph.pt` | PyG cumulative snapshots with normalized `edge_attr` |
| `gfp_variants/*.npy` | alternative GFP feature blocks (win48, win120, lc10, rich), row-aligned |

## 0. Setup

In [1]:
import json
import pickle

import numpy as np
import pandas as pd
import torch

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

results = []   # (check name, passed, detail) — summarised at the end

def check(name, passed, detail=''):
    results.append((name, bool(passed), detail))
    print(f'{"PASS" if passed else "FAIL"}  {name}  {detail}')

N_EDGES = 5_077_237          # transactions after truncation at 2022-09-10
N_LAUND = 4_522              # laundering transactions among them
N_NODES = 515_070            # accounts appearing in those transactions
T1, T2  = 3_046_342, 4_061_789   # end of train / end of val (60% / 80%)

## 1. Raw Inputs

Three files from Kaggle. `HI-Small_Trans.csv` repeats the header name `Account`, so it is read
with explicit column names. Only counts are checked here — the raw data is never modified.

In [2]:
raw_cols = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account',
            'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency',
            'Payment Format', 'Is Laundering']
raw = pd.read_csv('Data/HI-Small_Trans.csv', names=raw_cols, header=0, low_memory=False)
raw['Timestamp'] = pd.to_datetime(raw['Timestamp'])
print(f'raw transactions: {len(raw):,}   laundering: {raw["Is Laundering"].sum():,}')
display(raw.head(3))

kept = raw[raw['Timestamp'] < '2022-09-11']
check('raw: rows kept after truncation', len(kept) == N_EDGES, f'{len(kept):,}')
check('raw: laundering kept after truncation', kept['Is Laundering'].sum() == N_LAUND,
      f'{kept["Is Laundering"].sum():,}')
raw_accounts = set(kept['From Account']) | set(kept['To Account'])
check('raw: unique accounts after truncation', len(raw_accounts) == N_NODES, f'{len(raw_accounts):,}')
del raw, kept

raw transactions: 5,078,345   laundering: 5,177


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0


PASS  raw: rows kept after truncation  5,077,237
PASS  raw: laundering kept after truncation  4,522


PASS  raw: unique accounts after truncation  515,070


In [3]:
accounts = pd.read_csv('Data/HI-Small_accounts.csv', low_memory=False)
print(f'accounts file: {len(accounts):,} rows, {accounts["Account Number"].nunique():,} unique account numbers')
display(accounts.head(3))

patterns = open('Data/HI-Small_Patterns.txt').read()
n_attempts = patterns.count('BEGIN LAUNDERING ATTEMPT')
print(f'patterns file: {n_attempts} laundering attempts')
print(patterns[:400])
check('patterns: 370 annotated attempts', n_attempts == 370)

accounts file: 518,581 rows, 518,573 unique account numbers


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397


patterns file: 370 laundering attempts
BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
2
PASS  patterns: 370 annotated attempts  


## 2. `feature_meta.json` — the contract

Lists exactly which columns form each feature group, the dimensions the models must expect,
and the ablation grid.

In [4]:
meta = json.load(open('Data/feature_meta.json'))
for k, v in meta.items():
    print(f'{k:<24}', v if not isinstance(v, list) or len(v) < 8 else f'{len(v)} columns: {v[:4]} ... {v[-2:]}')

check('meta: BASE 20 / GFP 61 / EDGE 81 / NODE 6',
      (len(meta['BASE_EDGE_COLS']), len(meta['GFP_FEAT_COLS']), meta['EDGE_DIM'], meta['NODE_DIM']) == (20, 61, 81, 6))
check('meta: EDGE = BASE + GFP in that order',
      meta['EDGE_FEAT_COLS'] == meta['BASE_EDGE_COLS'] + meta['GFP_FEAT_COLS'])
check('meta: bank risk lives on edges, not nodes',
      {'Src_Bank_Risk', 'Dst_Bank_Risk'} <= set(meta['BASE_EDGE_COLS'])
      and not any('Bank_Risk' in c for c in meta['NODE_FEAT_COLS']))

BASE, GFP, EDGE, NODE = (meta['BASE_EDGE_COLS'], meta['GFP_FEAT_COLS'],
                         meta['EDGE_FEAT_COLS'], meta['NODE_FEAT_COLS'])

BASE_EDGE_COLS           20 columns: ['Amount_Log', 'Struct_Band', 'Hour_Sin', 'Hour_Cos'] ... ['PayFmt_Reinvestment', 'PayFmt_Wire']
GFP_FEAT_COLS            61 columns: ['scatter-gather_bins_2-3', 'scatter-gather_bins_3-5', 'scatter-gather_bins_5-inf', 'temp-cycle_bins_2-3'] ... ['dest_skew_amt_in', 'dest_kurtosis_amt_in']
EDGE_FEAT_COLS           81 columns: ['Amount_Log', 'Struct_Band', 'Hour_Sin', 'Hour_Cos'] ... ['dest_skew_amt_in', 'dest_kurtosis_amt_in']
NODE_FEAT_COLS           ['EntityType_Corporation', 'EntityType_Country', 'EntityType_Direct', 'EntityType_Individual', 'EntityType_Partnership', 'EntityType_Sole']
EDGE_DIM                 81
NODE_DIM                 6
truncation               kept Timestamp < 2022-09-11
bank_risk_smoothing_m    200
GBT_ROW                  edge features + src node features + dst node features
dropped_vs_LI_pipeline   ['Is_ACH', 'Currency_Mismatch', 'Bank_ID_Norm']
ABLATIONS                {'A_node_structure': [], 'B_plus_base': 'BASE_EDGE_COL

## 3. `edge_features.csv` — one row per transaction

Columns: `src_account`, `dst_account`, `label`, `Timestamp`, then the 20 baseline and 61 GFP
features. Values are **raw** (normalization happens later, only inside the graphs).
Rows are in time order — the first 60% are the training period.

In [5]:
edge = pd.read_csv('Data/edge_features.csv', low_memory=False)
edge['Timestamp'] = pd.to_datetime(edge['Timestamp'], format='mixed')
print(f'shape: {edge.shape}')
display(edge.head(3))

check('edge csv: row count', len(edge) == N_EDGES, f'{len(edge):,}')
check('edge csv: laundering count', edge['label'].sum() == N_LAUND, f'{edge["label"].sum():,}')
check('edge csv: columns = 4 metadata + EDGE_FEAT_COLS', list(edge.columns) == ['src_account', 'dst_account', 'label', 'Timestamp'] + EDGE)
check('edge csv: time-sorted', edge['Timestamp'].is_monotonic_increasing)
check('edge csv: no NaN in features', not edge[EDGE].isna().any().any())
check('edge csv: covers 2022-09-01 .. 09-10', str(edge['Timestamp'].min().date()) == '2022-09-01'
      and str(edge['Timestamp'].max().date()) == '2022-09-10')

shape: (5077237, 85)


,src_account,dst_account,label,Timestamp,Amount_Log,Struct_Band,Hour_Sin,Hour_Cos,DayOfWeek_Sin,DayOfWeek_Cos,Is_Weekend,Is_Self_Loop,Same_Bank,Dt_Src_Log,Dt_Dst_Log,...,dest_skew_amt_out,dest_kurtosis_amt_out,dest_fan_in,dest_deg_in,dest_ratio_in,dest_avg_ts_in,dest_sum_ts_in,dest_var_ts_in,dest_skew_ts_in,dest_kurtosis_ts_in,dest_avg_amt_in,dest_sum_amt_in,dest_var_amt_in,dest_skew_amt_in,dest_kurtosis_amt_in
0,8000F4670,8000F4670,0,2022-09-01,9.594008,0.0,0.0,1.0,0.433884,-0.900969,0.0,1.0,1.0,13.66933,13.66933,...,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,14675.57,14675.57,0.0,0.0,0.0
1,8005DFEB0,8005DFEB0,0,2022-09-01,6.800582,0.0,0.0,1.0,0.433884,-0.900969,0.0,1.0,1.0,13.66933,13.66933,...,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,897.37,897.37,0.0,0.0,0.0
2,8000F6850,8000F6850,0,2022-09-01,11.512805,0.0,0.0,1.0,0.433884,-0.900969,0.0,1.0,1.0,13.66933,13.66933,...,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,99986.94,99986.94,0.0,0.0,0.0


PASS  edge csv: row count  5,077,237
PASS  edge csv: laundering count  4,522
PASS  edge csv: columns = 4 metadata + EDGE_FEAT_COLS  
PASS  edge csv: time-sorted  


PASS  edge csv: no NaN in features  
PASS  edge csv: covers 2022-09-01 .. 09-10  


### 3.1 Baseline features — what they look like

In [6]:
display(edge[BASE].describe().T.round(4))

# binary flags must be 0/1, cyclical encodings inside [-1, 1], bank risk a probability
flags = ['Struct_Band', 'Is_Weekend', 'Is_Self_Loop', 'Same_Bank'] + [c for c in BASE if c.startswith('PayFmt_')]
check('baseline: flags are binary', edge[flags].isin([0, 1]).all().all())
check('baseline: cyclical encodings in [-1, 1]',
      edge[['Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos']].abs().max().max() <= 1 + 1e-6)
check('baseline: one payment format per row', (edge[[c for c in BASE if c.startswith('PayFmt_')]].sum(axis=1) == 1).all())
check('baseline: bank risk in [0, 1]', edge[['Src_Bank_Risk', 'Dst_Bank_Risk']].min().min() >= 0
      and edge[['Src_Bank_Risk', 'Dst_Bank_Risk']].max().max() <= 1)

,count,mean,std,min,25%,50%,75%,max
Amount_Log,5077237.0,6.9284,2.8431,0.0001,5.0322,6.7603,8.5464,24.0939
Struct_Band,5077237.0,0.0092,0.0957,0.0000,0.0000,0.0000,0.0000,1.0000
Hour_Sin,5077237.0,0.0006,0.6758,-1.0000,-0.7071,0.0000,0.7071,1.0000
Hour_Cos,5077237.0,0.0867,0.7319,-1.0000,-0.7071,0.2588,0.8660,1.0000
DayOfWeek_Sin,5077237.0,0.0713,0.5980,-0.9749,-0.4339,0.4339,0.4339,0.9749
DayOfWeek_Cos,5077237.0,-0.3932,0.6948,-0.9010,-0.9010,-0.9010,-0.2225,1.0000
Is_Weekend,5077237.0,0.1227,0.3281,0.0000,0.0000,0.0000,0.0000,1.0000
Is_Self_Loop,5077237.0,0.1164,0.3207,0.0000,0.0000,0.0000,0.0000,1.0000
Same_Bank,5077237.0,0.1361,0.3429,0.0000,0.0000,0.0000,0.0000,1.0000
Dt_Src_Log,5077237.0,7.8875,3.8424,0.0000,5.7071,7.2306,10.9768,13.6693


PASS  baseline: flags are binary  
PASS  baseline: cyclical encodings in [-1, 1]  


PASS  baseline: one payment format per row  
PASS  baseline: bank risk in [0, 1]  


### 3.2 Bank risk target encoding — is it really leakage-free?

Two properties must hold by construction:
- **val/test rows** of the same bank all carry one frozen value (fit on the training window);
- **train rows** use only *earlier* transactions: the very first train transaction of every bank
  must equal the global prior (no history yet).

In [7]:
raw_banks = pd.read_csv('Data/HI-Small_Trans.csv', names=raw_cols, header=0,
                        usecols=['Timestamp', 'From Bank'], low_memory=False)
raw_banks['Timestamp'] = pd.to_datetime(raw_banks['Timestamp'])
raw_banks = raw_banks[raw_banks['Timestamp'] < '2022-09-11'].sort_values('Timestamp', kind='stable')
edge['src_bank'] = raw_banks['From Bank'].to_numpy()   # same order as edge_features (both time-sorted, stable)
del raw_banks

is_train = np.arange(len(edge)) < T1
prior = edge.loc[is_train, 'label'].mean()

# val/test: one value per bank
per_bank_vals = edge.loc[~is_train].groupby('src_bank')['Src_Bank_Risk'].nunique()
check('bank TE: val/test rows frozen (one value per bank)', (per_bank_vals == 1).all(),
      f'{(per_bank_vals > 1).sum()} banks with >1 value')

# train: first transaction of every bank == prior (nothing earlier to learn from)
first_train = edge.loc[is_train].groupby('src_bank').head(1)
check('bank TE: first train txn of each bank == prior (causal)',
      np.allclose(first_train['Src_Bank_Risk'], prior, atol=1e-6), f'prior={prior:.6f}')

# a worked example: one busy bank's risk evolving through the training period
busy = edge.loc[is_train, 'src_bank'].value_counts().index[0]
ex = edge.loc[is_train & (edge['src_bank'] == busy), ['Timestamp', 'label', 'Src_Bank_Risk']]
print(f'Example — bank {busy}: risk over its first rows (starts at prior, moves with history):')
display(ex.iloc[[0, 1, 2, 100, 1000, 5000, -1]])
edge = edge.drop(columns='src_bank')

PASS  bank TE: val/test rows frozen (one value per bank)  0 banks with >1 value


PASS  bank TE: first train txn of each bank == prior (causal)  prior=0.000754
Example — bank 70: risk over its first rows (starts at prior, moves with history):


,Timestamp,label,Src_Bank_Risk
116,2022-09-01 00:00:00,0,0.000754
133,2022-09-01 00:00:00,0,0.000750
150,2022-09-01 00:00:00,0,0.000747
5548,2022-09-01 00:00:00,0,0.000503
50866,2022-09-01 00:04:00,0,0.002626
256846,2022-09-01 00:23:00,0,0.001952
3046337,2022-09-06 13:34:00,0,0.001441


### 3.3 GFP features — what they look like

Pattern histograms are small integer counts; vertex statistics are heavy-tailed (they get
log/clip/scaled inside the graphs). GFP counts the current edge itself, so an account's
first-ever transaction has `source_deg_out = 1`, except for the bounded batch-128 exposure.

In [8]:
pattern_cols = [c for c in GFP if '_bins_' in c]
vertex_cols  = [c for c in GFP if c not in pattern_cols]
print('pattern histogram columns:')
display(edge[pattern_cols].describe().T[['mean', 'max']].round(5))
print('vertex statistics (first 8):')
display(edge[vertex_cols[:8]].describe().T.round(3))

first_txn = (edge.groupby('src_account').cumcount() == 0).to_numpy()
deg = edge.loc[first_txn, 'source_deg_out']
check('GFP: first-ever transactions see (almost) only themselves',
      deg.max() < 128 and (deg == 1).mean() > 0.98, f'{(deg == 1).mean():.2%} exactly 1, max {deg.max():.0f}')
check('GFP: pattern bins are non-negative integers',
      (edge[pattern_cols] >= 0).all().all() and np.allclose(edge[pattern_cols], edge[pattern_cols].round()))

pattern histogram columns:


,mean,max
scatter-gather_bins_2-3,0.00017,14.0
scatter-gather_bins_3-5,0.00000,1.0
scatter-gather_bins_5-inf,0.00000,0.0
temp-cycle_bins_2-3,0.00022,1.0
temp-cycle_bins_3-5,0.00000,2.0
temp-cycle_bins_5-inf,0.00000,1.0
lc-cycle_bins_2-3,0.00022,1.0
lc-cycle_bins_3-5,0.00001,2.0
lc-cycle_bins_5-inf,0.00000,1.0


vertex statistics (first 8):


,count,mean,std,min,25%,50%,75%,max
source_fan_out,5077237.0,3.661230e+02,1.539579e+03,1.000,1.000,2.000000e+00,4.000000e+00,1.254100e+04
source_deg_out,5077237.0,8.519000e+02,3.591388e+03,1.000,2.000,5.000000e+00,9.000000e+00,2.638900e+04
source_ratio_out,5077237.0,2.267000e+00,1.192000e+00,1.000,1.333,2.000000e+00,2.800000e+00,1.400000e+01
source_avg_ts_out,5077237.0,3.482834e+05,2.654344e+05,0.000,75024.000,3.697614e+05,5.885467e+05,8.639400e+05
source_sum_ts_out,5077237.0,3.291028e+08,1.657223e+09,0.000,313680.000,1.296780e+06,3.849300e+06,1.657261e+10
source_var_ts_out,5077237.0,4.691199e+08,4.584501e+08,0.000,0.000,4.375608e+08,7.500896e+08,1.868833e+09
source_skew_ts_out,5077237.0,9.200000e-02,5.400000e-01,-3.276,-0.007,0.000000e+00,2.890000e-01,2.175800e+01
source_kurtosis_ts_out,5077237.0,1.298000e+00,4.060000e+00,-4690.263,0.000,1.431000e+00,1.817000e+00,7.401300e+01


PASS  GFP: first-ever transactions see (almost) only themselves  99.02% exactly 1, max 9


PASS  GFP: pattern bins are non-negative integers  


### 3.4 Signal sanity — laundering vs legitimate means

Not a correctness check, but the quickest way to see that the features carry information.

In [9]:
show = ['Amount_Log', 'Struct_Band', 'Same_Bank', 'Dt_Src_Log', 'Dt_Dst_Log',
        'Src_Bank_Risk', 'Dst_Bank_Risk', 'PayFmt_ACH',
        'temp-cycle_bins_2-3', 'lc-cycle_bins_3-5', 'source_fan_out', 'dest_deg_in']
legit = edge.loc[edge['label'] == 0, show].mean()
laund = edge.loc[edge['label'] == 1, show].mean()
display(pd.DataFrame({'legit mean': legit, 'laundering mean': laund,
                      'ratio': (laund / (legit + 1e-9)).round(2)}).sort_values('ratio', ascending=False))

,legit mean,laundering mean,ratio
lc-cycle_bins_3-5,0.000003,0.005750,2242.70
temp-cycle_bins_2-3,0.000187,0.031402,167.50
PayFmt_ACH,0.117464,0.846528,7.21
Struct_Band,0.009229,0.030296,3.28
Src_Bank_Risk,0.000632,0.001333,2.11
Dst_Bank_Risk,0.000620,0.001170,1.89
source_fan_out,365.918526,595.444715,1.63
Amount_Log,6.927088,8.415846,1.21
Dt_Dst_Log,8.591370,9.361882,1.09
dest_deg_in,3.696710,3.664529,0.99


## 4. `node_features.csv` — one row per account

In [10]:
node = pd.read_csv('Data/node_features.csv', low_memory=False)
print(f'shape: {node.shape}')
display(node.head(5))
print(node[NODE].sum().rename('accounts per entity type'))

check('nodes: columns = account_id + NODE_FEAT_COLS', list(node.columns) == ['account_id'] + NODE)
check('nodes: unique account ids', node['account_id'].is_unique)
check('nodes: exactly one entity type per account', (node[NODE].sum(axis=1) == 1).all())
graph_accounts = set(edge['src_account']) | set(edge['dst_account'])
covered = graph_accounts <= set(node['account_id'])
check('nodes: every graph account has node features', covered,
      f'{len(graph_accounts - set(node["account_id"])):,} missing')

shape: (518573, 7)


,account_id,EntityType_Corporation,EntityType_Country,EntityType_Direct,EntityType_Individual,EntityType_Partnership,EntityType_Sole
0,80B779D80,0,0,0,0,0,1
1,809D86900,1,0,0,0,0,0
2,80812BE00,0,0,0,0,1,0
3,81047F300,1,0,0,0,0,0
4,80BD8CF00,1,0,0,0,0,0


EntityType_Corporation    172347
EntityType_Country          6692
EntityType_Direct             67
EntityType_Individual        740
EntityType_Partnership    189680
EntityType_Sole           149047
Name: accounts per entity type, dtype: int64
PASS  nodes: columns = account_id + NODE_FEAT_COLS  
PASS  nodes: unique account ids  
PASS  nodes: exactly one entity type per account  


PASS  nodes: every graph account has node features  0 missing


## 5. `account_to_idx.pkl` and `standard_scaler.pkl`

In [11]:
account_to_idx = pickle.load(open('Data/account_to_idx.pkl', 'rb'))
print('account_to_idx sample:', list(account_to_idx.items())[:3])
check('account_to_idx: one id per graph account', len(account_to_idx) == N_NODES, f'{len(account_to_idx):,}')
check('account_to_idx: ids are 0..N-1', set(account_to_idx.values()) == set(range(N_NODES)))
check('account_to_idx: covers all edge accounts', graph_accounts <= set(account_to_idx))

scaler = pickle.load(open('Data/standard_scaler.pkl', 'rb'))
for group in ['log_group', 'std_group']:
    g = scaler[group]
    print(f'{group}: {len(g["cols"])} cols, log1p={g["log"]}, e.g. {g["cols"][:3]}')
print(f'pattern_cols (untouched): {len(scaler["pattern_cols"])}')
check('scaler: groups partition the 61 GFP columns',
      sorted(scaler['log_group']['cols'] + scaler['std_group']['cols'] + scaler['pattern_cols']) == sorted(GFP))

account_to_idx sample: [('8000F4670', 0), ('8005DFEB0', 1), ('8000F6850', 2)]
PASS  account_to_idx: one id per graph account  515,070
PASS  account_to_idx: ids are 0..N-1  
PASS  account_to_idx: covers all edge accounts  


log_group: 40 cols, log1p=True, e.g. ['source_fan_out', 'source_deg_out', 'source_avg_ts_out']
std_group: 12 cols, log1p=False, e.g. ['source_ratio_out', 'source_skew_ts_out', 'source_skew_amt_out']
pattern_cols (untouched): 9
PASS  scaler: groups partition the 61 GFP columns  


## 6. GFP variants — `gfp_variants/*.npy`

Alternative GFP blocks, row-aligned with `edge_features.csv`. `lc10` changes only the cycle
length, so its scatter-gather and temp-cycle columns must be **bit-identical** to V0 — a strong
proof of alignment across independent runs. The long-window variants must *differ* from V0.

In [12]:
variants = {}
for name in ['win48', 'win120', 'lc10', 'rich']:
    arr = np.load(f'Data/gfp_variants/{name}.npy', mmap_mode='r')
    cols = json.load(open(f'Data/gfp_variants/{name}_cols.json'))
    variants[name] = (arr, cols)
    print(f'{name:<7} shape={arr.shape}  first cols: {cols[:3]}')
    check(f'variant {name}: rows aligned + no NaN',
          arr.shape[0] == N_EDGES and len(cols) == arr.shape[1] and not np.isnan(np.asarray(arr[:1000])).any())

def block(name, wanted):
    arr, cols = variants[name]
    return np.asarray(arr[:, [cols.index(c) for c in wanted]])

sg_tc = [c for c in pattern_cols if not c.startswith('lc-cycle')]
tc    = [c for c in pattern_cols if c.startswith('temp-cycle')]
check('variants: lc10 sg/tc block == V0 (bit-identical alignment)',
      np.array_equal(block('lc10', sg_tc), edge[sg_tc].to_numpy(dtype=np.float32)))
check('variants: win48 temp-cycle differs from V0 (window took effect)',
      not np.array_equal(block('win48', tc), edge[tc].to_numpy(dtype=np.float32)))
check('variants: win120 differs from win48', not np.array_equal(block('win120', tc), block('win48', tc)))

# a glimpse of what a variant row looks like next to V0
i = int(np.flatnonzero(edge['temp-cycle_bins_3-5'].to_numpy() > 0)[0])
print(f'\nrow {i} — temp-cycle bins, V0 vs win120:')
print('  V0    :', edge.loc[i, tc].to_numpy())
print('  win120:', block('win120', tc)[i])

win48   shape=(5077237, 61)  first cols: ['scatter-gather_bins_2-3', 'scatter-gather_bins_3-5', 'scatter-gather_bins_5-inf']
PASS  variant win48: rows aligned + no NaN  
win120  shape=(5077237, 61)  first cols: ['scatter-gather_bins_2-3', 'scatter-gather_bins_3-5', 'scatter-gather_bins_5-inf']
PASS  variant win120: rows aligned + no NaN  
lc10    shape=(5077237, 61)  first cols: ['scatter-gather_bins_2-3', 'scatter-gather_bins_3-5', 'scatter-gather_bins_5-inf']
PASS  variant lc10: rows aligned + no NaN  
rich    shape=(5077237, 101)  first cols: ['fan_in_bins_2-4', 'fan_in_bins_4-8', 'fan_in_bins_8-13']
PASS  variant rich: rows aligned + no NaN  


PASS  variants: lc10 sg/tc block == V0 (bit-identical alignment)  


PASS  variants: win48 temp-cycle differs from V0 (window took effect)  


PASS  variants: win120 differs from win48  

row 1592135 — temp-cycle bins, V0 vs win120:
  V0    : [1.0 1.0 0.0]
  win120: [1. 1. 0.]


## 7. Graph snapshots — `train/val/test_graph.pt`

PyG `Data` objects. `y = -1` marks **context** edges: present so the GNN can see history,
excluded from loss and metrics (`eval_mask = False`). Baseline columns of `edge_attr` are
untouched, so they must equal the CSV exactly; GFP vertex stats are normalized.

In [13]:
expected = {'train_graph': (T1, 0), 'val_graph': (T2, T1), 'test_graph': (N_EDGES, T2)}
laund = {}
for name, (n_edges, eval_start) in expected.items():
    g = torch.load(f'Data/{name}.pt', weights_only=False)
    print(f'\n{name}: {g}')
    y = g.y.numpy(); em = g.eval_mask.numpy()
    laund[name] = int((y[em] == 1).sum())
    print(f'  y values: {dict(zip(*np.unique(y, return_counts=True)))}')
    print(f'  evaluated edges: {em.sum():,}  laundering among them: {laund[name]:,} ({laund[name]/em.sum():.4%})')

    check(f'{name}: edge count', g.edge_index.shape[1] == n_edges, f'{g.edge_index.shape[1]:,}')
    check(f'{name}: eval_mask = rows from {eval_start:,}', em.sum() == n_edges - eval_start and not em[:eval_start].any())
    check(f'{name}: y == -1 exactly on context edges', ((y == -1) == ~em).all())
    check(f'{name}: dims x={tuple(g.x.shape)} edge_attr={tuple(g.edge_attr.shape)}',
          g.x.shape == (N_NODES, 6) and g.edge_attr.shape == (n_edges, 81))
    check(f'{name}: no NaN', not torch.isnan(g.edge_attr).any() and not torch.isnan(g.x).any())

    # edge_index must match the CSV through account_to_idx
    src = edge['src_account'].iloc[:n_edges].map(account_to_idx).to_numpy()
    dst = edge['dst_account'].iloc[:n_edges].map(account_to_idx).to_numpy()
    check(f'{name}: edge_index matches CSV accounts',
          np.array_equal(g.edge_index[0].numpy(), src) and np.array_equal(g.edge_index[1].numpy(), dst))
    # baseline columns are not normalized -> identical to the CSV
    check(f'{name}: baseline edge_attr == CSV',
          np.allclose(g.edge_attr[:, :20].numpy(), edge[BASE].iloc[:n_edges].to_numpy(dtype=np.float32), atol=1e-5))
    # labels of evaluated edges match the CSV
    check(f'{name}: evaluated labels == CSV', np.array_equal(y[em], edge['label'].iloc[eval_start:n_edges].to_numpy()))
    # normalized vertex stats: roughly standardised on the train portion
    vs_idx = [20 + GFP.index(c) for c in vertex_cols]
    tr = g.edge_attr[:T1][:, vs_idx].numpy()
    print(f'  train vertex stats after normalization: mean={tr.mean():.3f} std={tr.std():.3f}')
    del g

check('graphs: laundering conserved across splits (train + val + test = 4,522)',
      sum(laund.values()) == N_LAUND, str(laund))


train_graph: Data(x=[515070, 6], edge_index=[2, 3046342], edge_attr=[3046342, 81], y=[3046342], edge_time=[3046342], eval_mask=[3046342], num_nodes=515070)
  y values: {0: 3044045, 1: 2297}
  evaluated edges: 3,046,342  laundering among them: 2,297 (0.0754%)
PASS  train_graph: edge count  3,046,342
PASS  train_graph: eval_mask = rows from 0  
PASS  train_graph: y == -1 exactly on context edges  
PASS  train_graph: dims x=(515070, 6) edge_attr=(3046342, 81)  
PASS  train_graph: no NaN  


PASS  train_graph: edge_index matches CSV accounts  


PASS  train_graph: baseline edge_attr == CSV  
PASS  train_graph: evaluated labels == CSV  


  train vertex stats after normalization: mean=0.000 std=1.000



val_graph: Data(x=[515070, 6], edge_index=[2, 4061789], edge_attr=[4061789, 81], y=[4061789], edge_time=[4061789], eval_mask=[4061789], num_nodes=515070)
  y values: {-1: 3046342, 0: 1014365, 1: 1082}
  evaluated edges: 1,015,447  laundering among them: 1,082 (0.1066%)
PASS  val_graph: edge count  4,061,789
PASS  val_graph: eval_mask = rows from 3,046,342  
PASS  val_graph: y == -1 exactly on context edges  
PASS  val_graph: dims x=(515070, 6) edge_attr=(4061789, 81)  
PASS  val_graph: no NaN  


...


test_graph: Data(x=[515070, 6], edge_index=[2, 5077237], edge_attr=[5077237, 81], y=[5077237], edge_time=[5077237], eval_mask=[5077237], num_nodes=515070)
  y values: {-1: 4061789, 0: 1014305, 1: 1143}
  evaluated edges: 1,015,448  laundering among them: 1,143 (0.1126%)
PASS  test_graph: edge count  5,077,237
PASS  test_graph: eval_mask = rows from 4,061,789  
PASS  test_graph: y == -1 exactly on context edges  
PASS  test_graph: dims x=(515070, 6) edge_attr=(5077237, 81)  


PASS  test_graph: no NaN  


PASS  test_graph: edge_index matches CSV accounts  


PASS  test_graph: baseline edge_attr == CSV  
PASS  test_graph: evaluated labels == CSV  


  train vertex stats after normalization: mean=0.000 std=1.000
PASS  graphs: laundering conserved across splits (train + val + test = 4,522)  {'train_graph': 2297, 'val_graph': 1082, 'test_graph': 1143}


## 8. Summary

In [14]:
summary = pd.DataFrame(results, columns=['check', 'passed', 'detail'])
n_pass = summary['passed'].sum()
print(f'{n_pass} / {len(summary)} checks passed')
display(summary.style.map(lambda v: 'color: green' if v is True else ('color: red' if v is False else ''), subset=['passed']))
assert summary['passed'].all(), 'Some checks failed — see table above'

61 / 61 checks passed


,check,passed,detail
0,raw: rows kept after truncation,True,"5,077,237"
1,raw: laundering kept after truncation,True,"4,522"
2,raw: unique accounts after truncation,True,"515,070"
3,patterns: 370 annotated attempts,True,
4,meta: BASE 20 / GFP 61 / EDGE 81 / NODE 6,True,
5,meta: EDGE = BASE + GFP in that order,True,
6,"meta: bank risk lives on edges, not nodes",True,
7,edge csv: row count,True,"5,077,237"
8,edge csv: laundering count,True,"4,522"
9,edge csv: columns = 4 metadata + EDGE_FEAT_COLS,True,
